# Food Review Pipeline (Wongnai)

Notebook รันทั้ง pipeline บนข้อมูล Wongnai (`review_body` / `stars`) ผ่านสคริปต์ใน `scripts/`

**ต้องรันจาก root ของ repo** (`foods-review-classification/`)

## 1. Setup

สร้าง venv แล้วเปิด kernel จาก venv นั้น (แนะนำ):

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
pip install -e .
```

ถ้าไม่ `pip install -e .` สคริปต์ยังรันได้เพราะมี `scripts/_bootstrap.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "scripts" / "preprocess.py").exists():
    raise SystemExit(
        "Run this notebook from the repo root (foods-review-classification/). "
        f"Current cwd: {ROOT}"
    )

def run_script(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / script), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

print("Repo root:", ROOT)

In [ ]:
# Optional: install deps (skip if already done in your venv)
INSTALL_DEPS = False

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=ROOT, check=True)

## 2. Optional: check CUDA

In [ ]:
try:
    import torch

    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch not installed — baseline XGBoost will use CPU (--device auto).")

## 3. Paths (Wongnai)

In [ ]:
RAW_CSV = ROOT / "data" / "wongnai-restaurant-review_train.csv"
PROCESSED = ROOT / "data" / "wongnai_processed.parquet"
ARTIFACT_DIR = ROOT / "artifacts" / "notebook_baseline"
METRICS_JSON = ROOT / "reports" / "notebook_baseline_metrics.json"
SAMPLE_PARQUET = ROOT / "data" / "wongnai_sample_500.parquet"
SCORED_OUT = ROOT / "outputs" / "notebook_scored_sample.parquet"

# Set to an int for a quick demo; None = full dataset (~79k rows, slower)
DEMO_MAX_ROWS: int | None = 5000
SCORE_SAMPLE_ROWS = 500

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing Wongnai CSV: {RAW_CSV}")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
SCORED_OUT.parent.mkdir(parents=True, exist_ok=True)

print("Input:", RAW_CSV)
print("Processed:", PROCESSED)
print("Artifacts:", ARTIFACT_DIR)
print("Demo max_rows:", DEMO_MAX_ROWS)

## 4. Preprocess → parquet

In [ ]:
run_script(
    "preprocess.py",
    "--input", str(RAW_CSV),
    "--out", str(PROCESSED),
)

## 5. Train baseline (TF-IDF + XGBoost)

`--device auto` ใช้ GPU ถ้ามี CUDA; ถ้า VRAM เต็มให้เปลี่ยนเป็น `--device cpu`

In [ ]:
train_args = [
    "--input", str(PROCESSED),
    "--out_dir", str(ARTIFACT_DIR),
    "--device", "auto",
    "--n_jobs", "-1",
]
if DEMO_MAX_ROWS is not None:
    train_args.extend(["--max_rows", str(DEMO_MAX_ROWS)])

run_script("train_baseline_xgb.py", *train_args)

## 6. Evaluate

In [ ]:
run_script(
    "evaluate.py",
    "--input", str(PROCESSED),
    "--model_type", "baseline_xgb",
    "--artifact_dir", str(ARTIFACT_DIR),
    "--out", str(METRICS_JSON),
    "--n_jobs", "-1",
)

import json

print(json.dumps(json.loads(METRICS_JSON.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

## 7. Score + flag (small sample)

In [ ]:
import pandas as pd

sample_df = pd.read_parquet(PROCESSED).head(SCORE_SAMPLE_ROWS)
sample_df.to_parquet(SAMPLE_PARQUET, index=False)
print(f"Wrote {len(sample_df)} rows -> {SAMPLE_PARQUET}")

run_script(
    "score_and_flag.py",
    "--input", str(SAMPLE_PARQUET),
    "--model_type", "baseline_xgb",
    "--artifact_dir", str(ARTIFACT_DIR),
    "--out", str(SCORED_OUT),
    "--n_jobs", "-1",
)

scored = pd.read_parquet(SCORED_OUT)
cols = [c for c in scored.columns if c in (
    "text", "user_rating", "ai_expected_rating", "delta", "is_anomaly", "ai_hex_color"
)]
scored[cols].head(10)